In [1]:
import requests

# -------------------------------------------------------------------------
# LISTA TICKER HL
# -------------------------------------------------------------------------
hl_tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

# -------------------------------------------------------------------------
# OVERRIDES opzionali HL→Binance (se un ticker ha un nome diverso)
# -------------------------------------------------------------------------
OVERRIDES = {
    # esempio:
    # "SPX": "SPX500",   # diventerebbe SPX500USDT
}

def hl_to_binance_base(hl):
    return OVERRIDES.get(hl, hl)

# -------------------------------------------------------------------------
# SCARICA LISTA SYMBOL SPOT USDT DA BINANCE
# -------------------------------------------------------------------------
resp = requests.get("https://api.binance.com/api/v3/exchangeInfo", timeout=10)
resp.raise_for_status()
data = resp.json()

symbols = data["symbols"]

# simboli spot USDT → set di: SYMBOLUSDT
binance_usdt = {
    s["symbol"]
    for s in symbols
    if s.get("quoteAsset") == "USDT" and s.get("status") == "TRADING"
}

# -------------------------------------------------------------------------
# CHECK ESISTENZA
# -------------------------------------------------------------------------
exists = []
missing = []

for hl in hl_tickers:
    symbol = hl_to_binance_base(hl) + "USDT"
    if symbol in binance_usdt:
        exists.append(hl)
    else:
        missing.append(hl)

# -------------------------------------------------------------------------
# STAMPA SOLO LE LISTE
# -------------------------------------------------------------------------
print("\n=== TICKER HL PRESENTI SU BINANCE (spot USDT) ===")
print(exists)

print("\n=== TICKER HL NON PRESENTI SU BINANCE (spot USDT) ===")
print(missing)

print(f"\nTotale HL: {len(hl_tickers)}  |  Presenti: {len(exists)}  |  Assenti: {len(missing)}")



=== TICKER HL PRESENTI SU BINANCE (spot USDT) ===
['ATOM', 'REQ', 'CRV', 'SAGA', 'NEAR', 'MORPHO', 'MANTA', 'MOVE', 'XAI', 'ETC', 'DOGE', 'SOPH', 'CELO', 'MAV', 'SCR', 'COMP', 'GMT', 'SOL', 'IMX', 'JUP', 'RUNE', 'UMA', 'TRB', 'USTC', 'AIXBT', 'IOTA', 'VIRTUAL', 'ALGO', 'GMX', 'ANIME', 'BCH', 'BIO', 'NXPC', 'TNSR', 'HBAR', 'SNX', 'HYPER', 'SAND', 'BERA', 'GAS', 'LDO', 'ONDO', 'DYDX', 'FTT', 'TON', 'EIGEN', 'LTC', 'AAVE', 'OGN', 'SUI', 'MEME', 'FXS', 'NIL', 'CFX', 'ME', 'XRP', 'TIA', 'BNB', 'NOT', 'OM', 'TAO', 'OP', 'CAKE', 'AVAX', 'GALA', 'BOME', 'SUPER', 'SEI', 'BABY', 'STX', 'S', 'STG', 'RENDER', 'ENA', 'LINK', 'ARB', 'ARK', 'BIGTIME', 'BTC', 'ETH', 'RSR', 'BANANA', 'XLM', 'INJ', 'ENS', 'AR', 'DOT', 'ETHFI', 'PAXG', 'FIL', 'STRK', 'TRX', 'ZK', 'KAITO', 'PENGU', 'ORDI', 'INIT', 'APT', 'REZ', 'LAYER', 'ZEN', 'SUSHI', 'ADA', 'PEOPLE', 'PENDLE', 'APE', 'FET', 'PNUT', 'WIF', 'ACE', 'TRUMP', 'NEO', 'JTO', 'YGG', 'ZRO', 'WLD', 'W', 'BLUR', 'UNI', 'DYM', 'MINA', 'POLYX', 'POL', 'IO', 'TURBO'

In [2]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from typing import Optional, Set, List, Dict, Any

API_URL = "https://api.hyperliquid.xyz/info"

# -------------------------------------------------------------------------
# LISTA TICKER
# -------------------------------------------------------------------------
tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

# -------------------------------------------------------------------------
# FUNZIONI DI SUPPORTO
# -------------------------------------------------------------------------

def safe_post(payload: Dict[str, Any],
              max_retries: int = 5,
              base_delay: float = 1.0) -> Optional[requests.Response]:
    """
    POST verso /info con gestione rate limit (429) e retry con backoff.
    """
    attempt = 0
    delay = base_delay

    while True:
        try:
            r = requests.post(API_URL, json=payload, timeout=10)
            if r.status_code == 429:
                attempt += 1
                if attempt > max_retries:
                    print(
                        f"Rate limit esaurito per type={payload.get('type')} "
                        f"coin={payload.get('coin') or payload.get('req', {}).get('coin')}, rinuncio."
                    )
                    return None
                print(
                    f"Rate limited (429) per type={payload.get('type')} "
                    f"coin={payload.get('coin') or payload.get('req', {}).get('coin')}, "
                    f"retry fra {delay:.1f}s..."
                )
                time.sleep(delay)
                delay *= 2
                continue

            r.raise_for_status()
            return r

        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                print(
                    f"HTTP error definitivo per type={payload.get('type')} "
                    f"coin={payload.get('coin') or payload.get('req', {}).get('coin')}: {e}"
                )
                return None
            print(
                f"HTTP error per type={payload.get('type')} "
                f"coin={payload.get('coin') or payload.get('req', {}).get('coin')}: {e}, "
                f"retry fra {delay:.1f}s..."
            )
            time.sleep(delay)
            delay *= 2


def get_perp_universe() -> Optional[Set[str]]:
    """
    Restituisce l'insieme dei nomi dei perps (campo 'name' in 'universe')
    usando type: 'meta'. Se fallisce, restituisce None.
    """
    payload = {"type": "meta"}
    r = safe_post(payload)
    if r is None:
        print("Impossibile ottenere la meta; uso la lista ticker così com'è.")
        return None

    try:
        meta = r.json()
    except Exception as e:
        print(f"Errore parsing meta JSON: {e} | raw: {r.text[:200]}")
        return None

    universe = meta.get("universe", [])
    names = {c.get("name") for c in universe if isinstance(c, dict) and "name" in c}
    return names


def detect_listing_time_ms(coin: str, end_ms: int) -> Optional[int]:
    """
    Stima la listing del perp come timestamp del primo record disponibile
    in fundingHistory (più vecchio).
    """
    payload = {
        "type": "fundingHistory",
        "coin": coin,
        "startTime": 0,
        "endTime": end_ms,
    }

    r = safe_post(payload)
    if r is None:
        print(f"Impossibile detectare listing per {coin} (safe_post fallita).")
        return None

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error per listing {coin}: {e} | raw: {r.text[:200]}")
        return None

    if not isinstance(data, list) or len(data) == 0:
        return None

    first = data[0]
    t = first.get("time")
    if t is None:
        return None

    return int(t)


def fetch_oracle_all(
    coin: str,
    start_ms: int,
    end_ms: int,
    interval: str = "1h",
) -> pd.DataFrame:
    """
    Scarica le candele 1h per `coin` tra start_ms ed end_ms usando candleSnapshot
    e usa il CLOSE come proxy di oraclePx.

    NB: l'endpoint espone solo le ultime 5000 candele per coin; se la finestra
    eccede questo limite, l'inizio verrà troncato lato server.
    """
    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms,
        },
    }

    r = safe_post(payload)
    if r is None:
        print(f"Impossibile scaricare candleSnapshot per {coin}.")
        return pd.DataFrame()

    try:
        data = r.json()
    except Exception as e:
        print(f"JSON decode error candleSnapshot {coin}: {e} | raw: {r.text[:200]}")
        return pd.DataFrame()

    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame()

    df = pd.DataFrame(data)

    # Controllo colonne standard: t (open time), c (close)
    if "t" not in df.columns or "c" not in df.columns:
        print(f"Risposta inattesa per candleSnapshot {coin}, colonne: {df.columns.tolist()}")
        return pd.DataFrame()

    df["time"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    df["oraclePx"] = pd.to_numeric(df["c"], errors="coerce")

    df_full = df[["time", "oraclePx"]].copy()
    return df_full


# -------------------------------------------------------------------------
# MAIN
# -------------------------------------------------------------------------

def main():
    # Start globale minimo richiesto
    global_start = datetime(2025, 6, 5, tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)

    global_start_ms = int(global_start.timestamp() * 1000)
    end_ms = int(now.timestamp() * 1000)

    # 1) Universo perps da HL (come nel codice funding)
    perp_universe = get_perp_universe()
    if perp_universe is not None:
        valid_tickers = [t for t in tickers if t in perp_universe]
        missing = sorted(set(tickers) - perp_universe)
        if missing:
            print("Ticker non trovati in universe (probabilmente non perps o nomi diversi):")
            print(", ".join(missing))
    else:
        valid_tickers = tickers

    oracle_rows: List[Dict[str, Any]] = []

    for ticker in valid_tickers:
        print(f"\n=== {ticker} ===")

        # 2) Detect listing via fundingHistory
        listing_ms = detect_listing_time_ms(ticker, end_ms)
        if listing_ms is None:
            print(f"Nessun funding / listing non determinabile per {ticker} — skipped")
            time.sleep(0.2)
            continue

        start_ms = max(global_start_ms, listing_ms)

        print(
            f"Listing {ticker}: {datetime.fromtimestamp(listing_ms/1000, tz=timezone.utc)} "
            f"| start effettivo: {datetime.fromtimestamp(start_ms/1000, tz=timezone.utc)}"
        )

        # 3) Candle 1h su tutto l'intervallo, close come proxy dell'oraclePx
        df_o = fetch_oracle_all(ticker, start_ms, end_ms, interval="1h")

        if df_o.empty:
            print(f"Nessun dato di prezzo in range per {ticker} — skipped")
            time.sleep(0.2)
            continue

        for _, row in df_o.iterrows():
            oracle_rows.append({
                "perp": ticker,
                "time": row["time"],
                "oraclePx": row["oraclePx"],
            })

        # piccolo sleep per non spammare l'endpoint candles
        time.sleep(0.2)

    if not oracle_rows:
        print("Nessun dato di oracle/price scaricato.")
        return

    oracle_df = pd.DataFrame(oracle_rows)
    oracle_df = oracle_df.sort_values(["perp", "time"])

    out_name = "oracle_price.csv"
    oracle_df.to_csv(out_name, index=False)
    print(f"\n✔️ Saved → {out_name}")


if __name__ == "__main__":
    main()



=== ATOM ===
Listing ATOM: 2023-05-12 00:00:00.048000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== REQ ===
Listing REQ: 2023-10-11 14:00:00.116000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== CRV ===
Listing CRV: 2023-05-15 16:00:00.409000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== MAVIA ===
Listing MAVIA: 2024-02-06 23:00:00.019000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== SAGA ===
Listing SAGA: 2024-04-21 07:00:00.074000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== NEAR ===
Listing NEAR: 2023-11-01 17:00:00.031000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== MORPHO ===
Listing MORPHO: 2025-01-17 09:00:00.042000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== MANTA ===
Listing MANTA: 2024-01-18 22:00:00.307000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== MOVE ===
Listing MOVE: 2024-12-11 05:00:00.032000+00:00 | start effettivo: 2025-06-05 00:00:00+00:00

=== XAI ===
Listing XAI: 2024-01-15 18:00